# Cox survival link & survival-model-choice divergence

**NOT FOR CLINICAL USE.** Population/trial-level forward simulation only. Executed in CI (nbmake).

The spec (§2, §6) calls for **Cox and parametric** OS/PFS link models. The Cox proportional-hazards link uses a *nonparametric* tabulated baseline survival S₀(t): `S(t | x) = S0(t)^exp(beta·x)` — the baseline comes from data, not a closed-form distribution. It is marked non-default so it doesn't auto-collide with the parametric Weibull link; selecting it explicitly lets you ask a new question: **how much does the choice of survival model itself move the prediction?**

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import onkos

ds = onkos.load()
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}
t = np.linspace(0, 208, 417)
weibull = onkos.simulate(ds, 'resistance.claret_2009.tgi', context=ctx, drug_effect=1.0, t=t)
cox = onkos.simulate(ds, 'resistance.claret_2009.tgi', context=ctx, drug_effect=1.0, t=t,
                     survival_link='survival_link.nsclc_os_cox')
print('median OS  Weibull:', round(weibull.median_os, 1), '| Cox:', round(cox.median_os, 1))
spread = float(np.max(np.abs(weibull.os_curve - cox.os_curve)))
print('max OS-curve spread (survival-model choice):', round(spread, 3))
assert spread > 0.05

In [ ]:
bs = ds['survival_link.nsclc_os_cox'].structure['baseline_survival']
plt.plot(t, weibull.os_curve, label='Weibull-PH')
plt.plot(t, cox.os_curve, label='Cox-PH')
plt.plot(bs['times'], bs['survival'], 'o', label='Cox baseline S0(t)')
plt.axhline(0.5, ls=':', color='grey'); plt.ylim(0, 1.02)
plt.xlabel('weeks'); plt.ylabel('survival fraction'); plt.legend(); plt.title('NSCLC OS: same metric, two survival models');
# Cox curve is a valid survival function
assert abs(cox.os_curve[0] - 1.0) < 1e-6 and np.all(np.diff(cox.os_curve) <= 1e-9)

In [ ]:
# Auto-discovery still yields exactly the default OS (Weibull) + PFS links;
# the non-default Cox link is opt-in only.
auto = onkos.simulate(ds, 'resistance.claret_2009.tgi', context=ctx, drug_effect=1.0)
print('auto endpoints:', sorted(auto.survival))
assert set(auto.survival) == {'OS', 'PFS'}